In [ ]:
# Resolve repository imports and data paths when opened from notebooks/.
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "sae").is_dir():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Run this notebook from the KronSAE repository.")
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Keep the toy utilities' imports and checkpoint paths relative to toy_model/.
toy_model_dir = repo_root / "toy_model"
os.chdir(toy_model_dir)
sys.path.insert(0, str(toy_model_dir))


In [ ]:
import os
import random
import math
import torch
import einops

import torch.nn as nn
import numpy as np
import pandas as pd

import torch.nn.functional as F

from torch.utils.data import IterableDataset, DataLoader
from typing import Optional
import pyrallis
from pprint import pprint

from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

from utils import *
from config import *
from toy_models import *
from data import *

In [ ]:
class ImporanceWeightedMSE(nn.Module):
    def __init__(self):
        super(ImporanceWeightedMSE, self).__init__()

    def forward(self, predictions, targets, importance):
        sub_total = ((predictions - targets) ** 2).sum(0).flatten()
        return sum(sub_total * importance)

def explained_variance_score(y_target, y_pred):
    per_token_l2_loss_A = (y_pred.float() - y_target.float()).pow(2).sum(-1).squeeze()
    total_variance_A = (y_target.float() - y_target.float().mean(0)).pow(2).sum(-1).squeeze()
    explained_variance = (1 - per_token_l2_loss_A / total_variance_A).mean()
    return explained_variance


def train_autoencoder(
    ae: SymAE, cfg: SynthConfig, device: str = "cuda", importance_score: int = None, dataset = None
) -> dict:
    """
    Trains the autoencoder for cfg.ae_train_steps steps on the fly.
    Prints training MSE, no validation.
    """
    ae.to(device)
    optimizer = torch.optim.Adam(ae.parameters(), lr=cfg.ae_lr)
    if importance_score is not None:
        mse_loss = ImporanceWeightedMSE()
    else:
        mse_loss = nn.MSELoss()

    metrics_list = []

    dataloader = DataLoader(dataset, batch_size=None)
    dataloader_iter = iter(dataloader)

    pbar = tqdm(range(cfg.ae_train_steps))

    for step in pbar:
        x = next(dataloader_iter).to(device)
        optimizer.zero_grad()
        x_recon = ae(x)
        if importance_score is not None:
            importance = (importance_score ** torch.arange(0, x.shape[1])).to(device)
            loss = mse_loss(x_recon, x, 1.0)
        else:
            loss = mse_loss(x_recon, x)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            metrics = {
                "loss" : loss.item(),
                "ev": explained_variance_score(x, x_recon).item(),
                "step": step
            }   
    
            metrics_list.append(metrics)

        pbar.set_description(f"[AE] metrics: {metrics}")

    if cfg.save_checkpoint:
        os.makedirs(cfg.ckpt_dir, exist_ok=True)
        torch.save(ae.state_dict(), os.path.join(cfg.ckpt_dir, "sym_ae.pt"))

    return metrics_list

@torch.no_grad()
def make_decoder_weights_and_grad_unit_norm_(W_dec, dim=-1):
    W_dec_normed = W_dec / W_dec.norm(dim=dim, keepdim=True)
    W_dec_grad_proj = (W_dec.grad * W_dec_normed).sum(
        dim, keepdim=True
    ) * W_dec_normed
    W_dec.grad -= W_dec_grad_proj
    W_dec.data = W_dec_normed


def train_topk_sae(
    sae: nn.Module,
    ae: SymAE,
    cfg: SynthConfig,
    dataloader: DataLoader,
    device: str = "cuda",
):
    """
    Trains the top-k SAE on the latent codes from the SymAE,
    including an auxiliary penalty to revive dead latents.
    """
    sae.to(device)
    ae.to(device)
    sae.eval()  # freeze the symmetrical AE

    # initialize dead-feature tracking
    num_batches_not_active = torch.zeros(sae.dict_size, device=device)
    optimizer = torch.optim.Adam(sae.parameters(), lr=cfg.fe_lr)
    mse_loss = nn.MSELoss()

    dataloader_iter = iter(dataloader)
    pbar = tqdm(range(cfg.fe_train_steps))
    metrics_list = []

    for step in pbar:
        x = next(dataloader_iter).to(device)
        with torch.no_grad():
            h = ae.encode(x)

        optimizer.zero_grad()
        acts = sae.encode(h)
        h_pred = sae.decode(acts)

        # primary reconstruction loss
        recon_loss = mse_loss(h_pred, h)

        # identify dead features
        dead = num_batches_not_active >= cfg.n_batches_to_dead
        aux_loss = torch.tensor(0.0, device=device)
        if dead.any():
            # compute residual between h and its reconstruction
            residual = (h - h_pred).detach()
            # select topk_aux activations among dead features
            dead_acts = acts[:, dead]
            k_aux = min(cfg.topk_aux, dead.sum().item())
            top_aux = torch.topk(dead_acts, k_aux, dim=-1)
            mask = torch.zeros_like(dead_acts).scatter(-1, top_aux.indices, top_aux.values)
            # reconstruct only from those dead features
            Wd = sae.W_dec[dead]
            x_aux_pred = mask @ Wd
            # L2 penalty on residual match
            aux_loss = cfg.aux_penalty * F.mse_loss(x_aux_pred, residual)

        # total loss
        loss = recon_loss + aux_loss
        loss.backward()

        # clip grads and normalize decoder weight directions
        torch.nn.utils.clip_grad_norm_(sae.parameters(), max_norm=1.)
        make_decoder_weights_and_grad_unit_norm_(sae.W_dec)
        optimizer.step()

        # update dead-feature counters
        active_now = (acts.sum(0) > 0).float()
        num_batches_not_active = torch.where(active_now.bool(),
                                          torch.zeros_like(num_batches_not_active),
                                          num_batches_not_active + 1)

        # log metrics
        with torch.no_grad():
            ev = explained_variance_score(h, h_pred).item()
            dead_count = int((num_batches_not_active >= cfg.n_batches_to_dead).sum())
            metrics = {
                "loss": loss.item(),
                "recon_loss": recon_loss.item(),
                "aux_loss": aux_loss.item(),
                "ev": ev,
                "dead_features": dead_count,
                "step": step,
            }
            metrics_list.append(metrics)
        pbar.set_description(f"[SAE] metrics: {metrics}")

    if cfg.save_checkpoint:
        os.makedirs(cfg.ckpt_dir, exist_ok=True)
        torch.save(sae.state_dict(), os.path.join(cfg.ckpt_dir, "topk_sae.pt"))

    return metrics_list


def eval_sae(sae: nn.Module, ae: SymAE, eval_dataloader: DataLoader, eval_steps: int, device: str = "cuda"):
    ae.to(device)
    ae.eval() # freeze the SAE
    sae.to(device)
    sae.eval()  # freeze the symmetrical AE
    mse_loss = nn.MSELoss()

    dataloader_iter = iter(eval_dataloader)

    pbar = tqdm(range(eval_steps))
    total_loss = 0
    total_ev = 0
    statistics = {
        "acts_std": 0., 
        "non_zero_acts_mean": 0.,
        "acts_distribution" : torch.zeros(sae.dict_size),
        "acts_value_distribution" : 0.
    }

    for step in pbar:
        x = next(dataloader_iter).to(device)

        with torch.no_grad():
            h = ae.encode(x)
            acts_topk = sae.encode(h)
            h_pred = sae.decode(acts_topk)
            loss = mse_loss(h_pred, h)
            total_ev += explained_variance_score(h, h_pred).item()

            non_zero_acts = acts_topk[acts_topk > 1e-8]
            statistics["acts_std"] += non_zero_acts.std(-1).mean()
            statistics["non_zero_acts_mean"] += non_zero_acts.mean()
            statistics["acts_distribution"] += (acts_topk.cpu() > 1e-8).float().sum(0)
    
        total_loss += loss

        pbar.set_description(f"[SAE] loss={loss.item():.6f}")
    total_loss /= eval_steps
    total_ev /= eval_steps

    statistics["acts_std"] = statistics["acts_std"] / eval_steps
    statistics["non_zero_acts_mean"] = statistics["non_zero_acts_mean"] / eval_steps

    return total_loss, total_ev, statistics


def eval_ae(ae: SymAE, eval_dataloader: DataLoader, eval_steps: int, device: str = "cuda"):
    ae.to(device)
    ae.eval()
    mse_loss = nn.MSELoss()

    dataloader_iter = iter(eval_dataloader)

    pbar = tqdm(range(eval_steps))
    total_loss = 0
    total_ev = 0

    for step in pbar:
        x = next(dataloader_iter).to(device)
        with torch.no_grad():
            x_recon = ae(x)
            loss = mse_loss(x_recon, x)
            total_ev += explained_variance_score(x, x_recon).item()

        total_loss += loss
        pbar.set_description(f"[AE] loss={loss.item():.6f}")

    total_loss /= eval_steps
    total_ev /= eval_steps
    return total_loss, total_ev

In [ ]:
cfg = SynthConfig()
cfg.ae_train_steps = 20000
cfg.fe_train_steps = 1000
cfg.S = 8
cfg.corr_inter = 0.0
#cfg.corr_intra = 0.8
cfg.corr_intra = (0.85, 0.95)
cfg.topk_aux = 32
cfg.aux_penalty = 1/16
cfg.sae_train_batch_size = 256 #or 32
cfg.heavy_tail_scale = 1.
set_all_seeds(cfg.seed)  # reproducibility
device = "mps"  # forced to "cuda"
print("==== Starting Training ====")
print("** Full Config **")
pprint(asdict(cfg))

#sizes = [32 for _ in range(8)]
sizes = [32, 16, 48, 32, 32, 32, 48, 16]

mask = np.array([
    [True,  False,  False, False, False, False, True,  True],
    [False, True,   False, False, False, False, True,  True],
    [False, False,  True,  False, False, False, False, False],
    [False, False,  False, True , False, False, False, False],
    [False, False,  False, False, True,  False, False, False],
    [False, False,  False, False, False, True,  False, False],
    [True,  True,   False, False, False, False, True,  False],
    [True,  True,   False, False, False, False, False, True],
], dtype=bool)

# 1) Create SymAE
ae = SymAE(input_dim=cfg.N, latent_dim=cfg.latent_dim, is_relu=True)
print("\n** Symmetrical Autoencoder (Phase 1) **")
print(ae)
total_params_ae = sum(p.numel() for p in ae.parameters())
print(f"SymAE has {total_params_ae:,} parameters.")

# 2) Train SymAE
dataset_ae = OnTheFlySynthDataset(
    cfg,
    batch_size = cfg.batch_size,
    mask=mask,
    sizes=sizes, 
    eps=1e-1
    )
ae_metrics = train_autoencoder(ae, cfg, device=device, dataset=dataset_ae) ##### common data

# 3) Create SAE
topk_sae = TopKSAE(input_dim=cfg.latent_dim, dict_size=cfg.hidden_dim, topk=cfg.topk)
print("\n** Top-K SAE (Phase 2) **")
print(topk_sae)
total_params_topk_sae = sum(p.numel() for p in topk_sae.parameters())
print(f"TopKSAE has {total_params_topk_sae:,} parameters.")
 
kronsae = kronsae(input_dim=cfg.latent_dim, dict_size=cfg.hidden_dim, topk=cfg.topk, num_heads=4, m_keys = 2, n_keys = 32)
print("\n** kronsae (Phase 2) **")
print(kronsae)
total_params_kronsae = sum(p.numel() for p in kronsae.parameters())
print(f"kronsae has {total_params_kronsae:,} parameters.")

######
# 4) Train SAE
dataset = OnTheFlySynthDataset(
    cfg, 
    on_the_fly = False, 
    size=cfg.fe_train_steps,
    batch_size = cfg.sae_train_batch_size,
    mask=mask,
    sizes=sizes, eps=1e-1
    )
dataloader = DataLoader(dataset, batch_size=None)

topk_sae_metrics = train_topk_sae(topk_sae, ae, cfg, dataloader, device=device)
kronsae_metrics = train_topk_sae(kronsae, ae, cfg, dataloader, device=device)
######

print("Done!")

In [ ]:
setups = {
    "kronsae_h2_m4_n32": {"h":2, "m":4, "n":32},
    "kronsae_h4_m2_n32": {"h":4, "m":2, "n":32},
    # "kronsae_h4_m4_n16": {"h":4, "m":4, "n":16},
    # "kronsae_h4_m8_n8": {"h":4, "m":8, "n":8},
    # "kronsae_h8_m2_n16": {"h":8, "m":2, "n":16},
}


mask_list = [
    {
        "sizes": [32, 48, 16, 32, 32, 48, 16, 32 ],
        "mask": np.array([
            [True,  False,  False, False, False, False, True,  True],
            [False, True,   False, False, False, False, True,  True],
            [False, False,  False,  False, False, False, False, False],
            [False, False,  False, False , False, False, False, False],
            [False, False,  False, False, False,  False, False, False],
            [False, False,  False, False, False, False,  False, False],
            [True,  True,   False, False, False, False, True,  False],
            [True,  True,   False, False, False, False, False, True],
        ], dtype=bool)
    },
    {
        "sizes": [32, 16, 48, 32, 32, 32, 48, 16],
        "mask": np.array([
            [True,  True,  False, False, False, False, False,  False],
            [True, True,   True, False, False, False, False,  False],
            [False, True,  True,  True, False, False, False, False],
            [False, False,  True, True , True, False, False, False],
            [False, False,  False, True, True,  True, False, False],
            [False, False,  False, False, True, True,  True, False],
            [False,  False,   False, False, False, True, True,  True],
            [False,  False,   False, False, False, False, True, True],
        ], dtype=bool)
    },
    {
        "sizes": [32, 32, 32, 32, 32, 32, 32, 32],
        "mask": np.array([
            [False,  False, False, False, False,False, False, False],
            [False, False,  False, False, False,False, False, False],
            [False, False,  True,  True,  True, True,  False, False],
            [False, False,  True,  True , True, True,  False, False],
            [False, False,  True,  True,  True, True,  False, False],
            [False, False,  True,  True,  True, True,  False, False],
            [False,  False, False, False, False,False, False, False],
            [False,  False, False, False, False,False, False, False],
        ], dtype=bool)
    },
    {
        "sizes": [32, 32, 32, 32, 32, 32, 32, 32],
        "mask": np.array([
            [True,  False,  False, False, False, False, False, True],
            [False, True,   False, False, False, False, True, False],
            [False, False,  True,  False, False, True, False, False],
            [False, False,  False, True , True, False, False, False],
            [False, False,  False, True, True,  False, False, False],
            [False, False,  True, False, False, True,  False, False],
            [False, True,  False, False, False, False, True,  False],
            [True, False,  False, False, False, False, False, True],
        ], dtype=bool)
    },
    {
        "sizes": [32, 32, 32, 32, 32, 32, 32, 32],
        "mask": np.array([
            [False,  False,  False, False, False, False, False, True],
            [False, False,   False, False, False, False, True, False],
            [False, False,  False,  False, False, True, False, False],
            [False, False,  False, False , True, False, False, False],
            [False, False,  False, True, False,  False, False, False],
            [False, False,  True, False, False, False,  False, False],
            [False, True,  False, False, False, False, False,  False],
            [True, False,  False, False, False, False, False, False],
        ], dtype=bool)
    }
]


def get_trained_saes(setup, mask_setups, cfg, dataloader, ae, device):
    trained_saes = []
    for mask_setup in mask_setups:
        sizes = mask_setup["sizes"]
        mask = mask_setup["mask"]
        cfg = SynthConfig()
        cfg.ae_train_steps = 20000
        cfg.fe_train_steps = 1000
        cfg.S = 8
        cfg.corr_inter = 0.0
        #cfg.corr_intra = 0.8
        cfg.corr_intra = (0.9, 0.9)
        cfg.topk_aux = 32
        cfg.aux_penalty = 1/16
        cfg.sae_train_batch_size = 256 #or 32
        cfg.heavy_tail_scale = 1.
        set_all_seeds(cfg.seed)  # reproducibility
        device = "mps"  # forced to "cuda"

        # 1) Create SymAE
        ae = SymAE(input_dim=cfg.N, latent_dim=cfg.latent_dim, is_relu=True)

        # 2) Train SymAE
        dataset_ae = OnTheFlySynthDataset(
            cfg,
            batch_size = cfg.batch_size,
            mask=mask,
            sizes=sizes, 
            eps=1e-1
            )
        ae_metrics = train_autoencoder(ae, cfg, device=device, dataset=dataset_ae) ##### common data

        # 3) Create SAE
        topk_sae = TopKSAE(input_dim=cfg.latent_dim, dict_size=cfg.hidden_dim, topk=cfg.topk)
        
        kronsae1 = kronsae(input_dim=cfg.latent_dim, dict_size=cfg.hidden_dim, topk=cfg.topk, num_heads=setups["kronsae_h2_m4_n32"]["h"], m_keys=setups["kronsae_h2_m4_n32"]["m"], n_keys=setups["kronsae_h2_m4_n32"]["n"])
        
        kronsae2 = kronsae(input_dim=cfg.latent_dim, dict_size=cfg.hidden_dim, topk=cfg.topk, num_heads=setups["kronsae_h4_m2_n32"]["h"], m_keys=setups["kronsae_h4_m2_n32"]["m"], n_keys=setups["kronsae_h4_m2_n32"]["n"])

        ######
        # 4) Train SAE
        dataset = OnTheFlySynthDataset(
            cfg, 
            on_the_fly = False, 
            size=cfg.fe_train_steps,
            batch_size = cfg.sae_train_batch_size,
            mask=mask,
            sizes=sizes, eps=1e-1
            )
        dataloader = DataLoader(dataset, batch_size=None)

        topk_sae_metrics = train_topk_sae(topk_sae, ae, cfg, dataloader, device=device)
        kronsae1_metrics = train_topk_sae(kronsae1, ae, cfg, dataloader, device=device)
        kronsae2_metrics = train_topk_sae(kronsae2, ae, cfg, dataloader, device=device)

        trained_saes.append(
            {
                "dataset": dataset,
                "ae": ae,
                "topk": topk_sae,
                "kronsae_h2_m4_n32": kronsae1,
                "kronsae_h4_m2_n32": kronsae2,
            }
        )
    return trained_saes    

trained_saes = get_trained_saes(setups, mask_list,cfg, dataloader, ae, device)

In [ ]:
def direction_similarity(W_dec: torch.Tensor):
    W_norm = F.normalize(W_dec, p=2, dim=1)
    return W_norm @ W_norm.T

def pearson_correlation(W_dec: torch.Tensor):
    W_norm = F.layer_norm(W_dec, normalized_shape=(W_dec.size(1),), elementwise_affine=False)
    return W_norm @ W_norm.T

In [ ]:
import torch
from scipy.optimize import linear_sum_assignment

def find_one_to_one_mapping(set_A: torch.Tensor, set_B: torch.Tensor) -> torch.Tensor:
    cost_matrix = torch.cdist(set_A, set_B).detach().cpu().numpy()  # (n, m)
    
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    mapping = torch.full((set_A.shape[0],), -1, dtype=torch.long, device=set_A.device)
    mapping[row_ind] = torch.tensor(col_ind, dtype=torch.long, device=set_A.device)
    
    return mapping

In [ ]:
def rv_coefficient(C1, C2):
    num = np.trace(C1 @ C2)
    den = np.sqrt(np.trace(C1 @ C1) * np.trace(C2 @ C2))
    return num / den

def rv_permutation_pvalue(C1, C2, num_permutations=1000, seed=None):
    """
    Permutation test for RV coefficient significance.
    Returns observed RV and p-value.
    """
    rng = np.random.default_rng(seed)
    d = C1.shape[0]
    RV_obs = rv_coefficient(C1, C2)
    count = 0

    for _ in range(num_permutations):
        perm = rng.permutation(d)
        C2_perm = C2[perm][:, perm]        # permute both rows and cols
        RV_perm = rv_coefficient(C1, C2_perm)
        if RV_perm >= RV_obs:
            count += 1

    p_value = (count + 1) / (num_permutations + 1)
    return RV_obs, p_value


In [ ]:
import matplotlib.pyplot as plt

title_kwargs = dict(fontsize=10, fontweight="bold", pad=6)
cmap       = 'magma'
panel_labels = ['(a)', '(b)', '(b)', '(c)', "(d)"]
panel_labels = ['', '', '', '', ""]

take_first_n = 4
fig, axs = plt.subplots(take_first_n, 5, figsize=(15, 3 * take_first_n), dpi=250)
for j, set_to_vis in enumerate(trained_saes[:take_first_n]):
    ae_selected = set_to_vis["ae"]
    topk_selected = set_to_vis["topk"]
    dataset_selected = set_to_vis["dataset"]
    kronsae_h2_m4_n32 = set_to_vis["kronsae_h2_m4_n32"]
    kronsae_h4_m2_n32 = set_to_vis["kronsae_h4_m2_n32"]
    idxs = find_one_to_one_mapping(ae_selected.W_enc, topk_selected.W_dec)
    data_list = [
        (dataset_selected.L @ dataset_selected.L.T, 'Covariance Matrix', 'Covariance value'),
        (direction_similarity(topk_selected.W_dec).cpu().detach().numpy(), 'TopkSAE', 'Similarity'),
        (direction_similarity(topk_selected.W_dec[idxs]).cpu().detach().numpy(), 'Matсhed $W_{dec}$ TopkSAE', 'Similarity'),
        (direction_similarity(kronsae_h2_m4_n32.W_dec).cpu().detach().numpy(), 'KronSAE $(h=2,\\ m=4,\\ n=32)$', 'Similarity'),
        (direction_similarity(kronsae_h4_m2_n32.W_dec).cpu().detach().numpy(), 'KronSAE $(h=4,\\ m=2,\\ n=32)$', 'Similarity'),
    ]

    for ax, (data, title, cbar_lbl), lbl in zip(axs[j], data_list, panel_labels):
        #im = ax.imshow(data, cmap=cmap)
        im = ax.imshow(
            data.cpu().detach().numpy() if hasattr(data, "cpu") else data,
            cmap=cmap,
            interpolation="nearest",
            aspect="equal"
        )
        ax.set_aspect('equal')
        ax.axis('off')
        if j == 0:
            ax.set_title(title, **title_kwargs)

    RV_obs, p_val = rv_permutation_pvalue(direction_similarity(topk_selected.W_dec).cpu().detach().numpy(), (dataset_selected.L @ dataset_selected.L.T).cpu().numpy(), num_permutations=5000, seed=42)
    print(f"RV topksae = {RV_obs:.3f}, p‐value ≈ {p_val:.3e}")
    RV_obs, p_val = rv_permutation_pvalue(direction_similarity(topk_selected.W_dec[idxs]).cpu().detach().numpy(), (dataset_selected.L @ dataset_selected.L.T).cpu().numpy(), num_permutations=5000, seed=42)
    print(f"RV topksae matched = {RV_obs:.3f}, p‐value ≈ {p_val:.3e}")
    RV_obs, p_val = rv_permutation_pvalue(direction_similarity(kronsae_h2_m4_n32.W_dec).cpu().detach().numpy(), (dataset_selected.L @ dataset_selected.L.T).cpu().numpy(), num_permutations=5000, seed=42)
    print(f"RV kronsae_h2_m4_n32 = {RV_obs:.3f}, p‐value ≈ {p_val:.3e}")
    RV_obs, p_val = rv_permutation_pvalue(direction_similarity(kronsae_h4_m2_n32.W_dec).cpu().detach().numpy(), (dataset_selected.L @ dataset_selected.L.T).cpu().numpy(), num_permutations=5000, seed=42)
    print(f"RV kronsae_h4_m2_n32 = {RV_obs:.3f}, p‐value ≈ {p_val:.3e}")
    print("----")

cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Value', fontsize=12)

fig.savefig('figure_synth_all_appendix.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
from utils import *

import matplotlib.pyplot as plt 
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import TwoSlopeNorm

def graph_biases(bias, ax_obj):
  b = bias.clone().detach().cpu()
  colors = [(.4, 0, 1), (1, 1, 1), (1, .4, 0)]  # Purple -> White -> Orange
  n_bins = 100 
  cm = LinearSegmentedColormap.from_list("", colors, N=n_bins)
  
  norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
  
  ax_obj.imshow(b, cmap=cm, norm=norm)

  ax_obj.set_xticks([])
  ax_obj.set_yticks([])

def graph_weights(weights, bias):
    fig, axs = plt.subplots(1, 2, figsize=(7, 3.5)) # 1 row, 2 columns
    
    w = weights.clone().cpu().detach()
    to_graph = w.T @ w
    colors = [(.4, 0, 1), (1, 1, 1), (1, .4, 0)]  # Purple -> White -> Orange
    n_bins = 100 
    cm = LinearSegmentedColormap.from_list("", colors, N=n_bins)
    
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    
    axs[0].imshow(to_graph, cmap=cm, norm=norm)
    axs[0].set_xticks([])
    axs[0].set_yticks([])

    graph_biases(bias, axs[1])
    plt.subplots_adjust(left=0.0, right=1.4)
    plt.tight_layout()
    plt.show()

In [ ]:

def direction_similarity(W_dec):
    # Normalize each row (feature vector) to unit norm
    W_norm = F.normalize(W_dec, p=2, dim=1)
    # Compute cosine similarity (i.e., direction similarity)
    return (W_norm @ W_norm.T).cpu().detach().numpy()

def pearson_correlation(W_dec):
    # Center and scale each row to zero mean and unit variance using layer_norm
    W_norm = F.layer_norm(W_dec, normalized_shape=(W_dec.size(1),))
    # Compute Pearson correlation (equivalent to cosine similarity of normalized rows)
    return (W_norm @ W_norm.T).cpu().detach().numpy()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- 1. Prepare your data ---
cov = dataset.L @ dataset.L.T
sim1 = direction_similarity(topk_sae.W_dec).detach().cpu()
sim2 = direction_similarity(topk_sae.W_dec[idxs]).detach().cpu()
sim3 = direction_similarity(kronsae.W_dec).detach().cpu()


# Compute global vmin and vmax
all_data = np.concatenate([cov.ravel(), sim1.ravel(), sim2.ravel()])
vmin, vmax = all_data.min(), all_data.max()

# --- 2. Set up figure ---
num_plots = 4
fig, axs = plt.subplots(2, 2, figsize=(8, 8), dpi=250,
                        gridspec_kw={'wspace': 0.05, 'right': 0.85})

cmap = 'magma'
title_kwargs = dict(fontsize=14, fontweight='bold')
labels = ['(a)', '(b)', '(c)', '(d)']
labels = ['', '', '', '']
titles = [
    r'$Covariance Matrix$',
    r'$TopkSAE$',
    r'$Matched$ $TopkSAE$',
    r'$KronSAE\ (h=4,\ m=2,\ n=32)$',
]
data_list = [cov, sim1, sim2, sim3]

# Plot each with same vmin/vmax
for ax_num, data, title, lbl in zip([i for i in range(num_plots)], data_list, titles, labels):
    ax = axs[ax_num // 2, ax_num % 2]
    #ax = axs[ax_num]
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    im = ax.imshow(
            data.cpu().detach().numpy() if hasattr(data, "cpu") else data,
            cmap=cmap,
            interpolation="nearest",
            aspect="equal"
        )
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, **title_kwargs)
    ax.text(-0.05, 1.05, lbl, transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top')

# --- 3. Add a shared colorbar ---
# Create an axes for the colorbar on the right
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Pearson $r$', fontsize=12)

# --- 4. Final touches and save ---
fig.tight_layout(pad=2.0)
fig.savefig('figure_synth_covariance_2_2.pdf', format='pdf', bbox_inches='tight')


In [ ]:

# Example usage:
RV_obs, p_val = rv_permutation_pvalue(direction_similarity(topk_sae.W_dec), (dataset.L @ dataset.L.T).cpu().numpy(), num_permutations=5000, seed=42)
print(f"RV topksae = {RV_obs:.3f}, p‐value ≈ {p_val:.3e}")
RV_obs, p_val = rv_permutation_pvalue(direction_similarity(kronsae.W_dec), (dataset.L @ dataset.L.T).cpu().numpy(), num_permutations=5000, seed=42)
print(f"RV kronsae = {RV_obs:.3f}, p‐value ≈ {p_val:.3e}")